# SCP Colab Experiment

這個 notebook 用來在 Google Colab 上快速測試 SCP / GAIA pipeline。

注意：SCP 預設會呼叫本機 OpenAI-compatible / Ollama endpoint。Colab 上通常沒有本機 Ollama，所以需要設定遠端 endpoint，或把模型服務另外開在可連線的位置。

## 1. Colab Runtime

建議在 Colab 選擇：

- Runtime type: Python 3
- Hardware accelerator: GPU

如果只是測 CLI / dataset loader，可以不開 GPU；如果要測 embedding-delta query salience 或 HuggingFace model，建議開 GPU。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print('Python:', sys.version)
!nvidia-smi || true

## 2. 取得專案

二選一：

1. 設定 `REPO_URL`，讓 Colab 直接 clone。
2. 把專案放在 Google Drive，設定 `PROJECT_DIR` 指向該資料夾。

如果你的 repo 還沒公開，建議先把 SCP 資料夾放到 Google Drive。

In [ ]:
# Option A: clone from GitHub
REPO_URL = ''  # example: 'https://github.com/your-org/SCP.git'

# Option B: use Google Drive project folder
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/SCP'

if REPO_URL:
    PROJECT_DIR = Path('/content/SCP')
    if not PROJECT_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
elif USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
else:
    PROJECT_DIR = Path('/content/SCP')
    print('請設定 REPO_URL，或把專案上傳/掛載到', PROJECT_DIR)

print('PROJECT_DIR =', PROJECT_DIR)
if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    print('cwd =', Path.cwd())
    print('files =', [p.name for p in PROJECT_DIR.iterdir()][:20])

## 3. 安裝依賴

如果 repo 有 `requirements.txt`，會先安裝它；接著補上 Colab 實驗常用套件。

In [ ]:
assert PROJECT_DIR.exists(), 'PROJECT_DIR 不存在，請先完成上一格設定。'
os.chdir(PROJECT_DIR)

if Path('requirements.txt').exists():
    !pip install -q -r requirements.txt

!pip install -q python-dotenv datasets huggingface_hub transformers accelerate torch ddgs requests markdownify

## 4. 設定環境變數

請依你的環境修改。若 Colab 要呼叫遠端 Ollama / OpenAI-compatible endpoint，請填 `OLLAMA_BASE_URL`。

In [ ]:
# Model endpoint
os.environ['OLLAMA_BASE_URL'] = 'http://YOUR_REMOTE_HOST:11434/v1'
os.environ['OLLAMA_API_KEY'] = ''
os.environ['OLLAMA_TIMEOUT'] = '180'

# Default SLM aliases
os.environ['Nemotron_MODEL_ID'] = 'nemotron-mini:4b'
os.environ['Minicpm_MODEL_ID'] = 'minicpm3:4b'
os.environ['Qwen_MODEL_ID'] = 'qwen3:4b'
os.environ['Gemma_MODEL_ID'] = 'gemma3:4b'

# Search backend
os.environ['SEARCH_BACKEND'] = 'searxng'
os.environ['SEARXNG_URL'] = 'http://YOUR_SEARXNG_HOST:8080'
os.environ['SEARCH_SALIENCE_HF_MODEL'] = 'BAAI/bge-m3'

# Optional: HuggingFace token for GAIA dataset access
os.environ['HF_TOKEN'] = ''

for key in ['OLLAMA_BASE_URL', 'SEARCH_BACKEND', 'SEARXNG_URL', 'SEARCH_SALIENCE_HF_MODEL']:
    print(key, '=', os.environ.get(key, ''))

## 5. 基本 Import / Compile 檢查

In [ ]:
os.chdir(PROJECT_DIR)
!python -m compileall -q core benchmark tools run_gaia.py

from tools.search_tool import SearchTool
from tools.search_result_builder.evidence_searcher import EvidenceSearcher
print('Import OK')

## 6. SearchTool Smoke Test

`SearchTool` 現在只負責呼叫搜尋後端並回傳 raw normalized results。若你的 SearXNG endpoint 尚未設定，這格可能會失敗。

In [ ]:
tool = SearchTool(backend=os.environ.get('SEARCH_BACKEND', 'searxng'))
result = tool.run({'input': 'Taiwan capital', 'max_results': 3})
result

## 7. GAIA Smoke Test

如果模型 endpoint、search backend、GAIA dataset 都可用，可以跑一題。Colab 上第一次下載資料與模型會比較慢。

In [ ]:
!python run_gaia.py --level 1 --max-samples 1 --log-name colab_gaia_l1_smoke

## 8. 查看輸出

In [ ]:
from pathlib import Path
out_dir = Path('outputs/colab_gaia_l1_smoke')
print('exists:', out_dir.exists())
if out_dir.exists():
    for path in sorted(out_dir.rglob('*'))[:30]:
        print(path)

## 9. 備註

- EfficientRAG labeler model 若尚未訓練完成，`rag_labeler.py` 會先使用 deterministic fallback。
- Colab 若沒有遠端 Ollama / OpenAI-compatible endpoint，Stage 1 / Stage 2 agent 呼叫會失敗。
- SearchTool 已解耦，不會 rerank、不會 conditional fetch、不會建 evidence；這些工作都由 `tools/search_result_builder/` 負責。